In [0]:
%sql
-- Criação do primeiro catálogo (ex: Ambiente de Desenvolvimento)
CREATE CATALOG IF NOT EXISTS dev_catalog;

-- Criação do segundo catálogo (ex: Ambiente de Homologação/Testes)
CREATE CATALOG IF NOT EXISTS stage_catalog;

-- Criação do terceiro catálogo (ex: Ambiente de Produção)
CREATE CATALOG IF NOT EXISTS prod_catalog;


In [0]:
%sql
-- Criação do schema no catálogo de new_sample

CREATE SCHEMA IF NOT EXISTS new_sample.health_data;

/*
-- Criação do schema no catálogo de Desenvolvimento
CREATE SCHEMA IF NOT EXISTS dev_catalog.health_data;


-- Criação do schema no catálogo de Homologação
CREATE SCHEMA IF NOT EXISTS staging_catalog.health_data;

-- Criação do schema no catálogo de Produção
CREATE SCHEMA IF NOT EXISTS prod_catalog.health_data; 
*/


In [0]:
%sql
-- Criação do volume no catálogo new_sample
CREATE VOLUME IF NOT EXISTS new_sample.health_data.health;


/*
-- Criação do volume no catálogo de Desenvolvimento
CREATE VOLUME IF NOT EXISTS dev_catalog.default.health;

-- Criação do volume no catálogo de Homologação
CREATE VOLUME IF NOT EXISTS staging_catalog.default.health;

-- Criação do volume no catálogo de Produção
CREATE VOLUME IF NOT EXISTS prod_catalog.default.health; */


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from pyspark.sql.functions import when, col, current_timestamp

my_catalog = 'new_sample'
##
## CSV -> Bronze
##

# Set the path to the CSV file in the DEV volume
csv_path = f'/Volumes/{my_catalog}/health_data/health'

# Define the schema for the CSV file
health_csv_schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("PII", StringType(), True),
    StructField("date", DateType(), True),
    StructField("HighCholest", IntegerType(), True),
    StructField("HighBP", DoubleType(), True),
    StructField("BMI", DoubleType(), True),
    StructField("Age", DoubleType(), True),
    StructField("Education", DoubleType(), True),
    StructField("income", IntegerType(), True)
])


# Ingest the CSV file and add metadata columns for the ingested data
health_raw = (
    spark
    .read
    .format("csv") 
    .option("header", "true")             # Use the header row for column names
    .schema(health_csv_schema)            # Apply the defined schema
    .load(csv_path)                       # Load the CSV data
    .select(
        "*",
        "_metadata.file_name",                        # Include file name from metadata
        "_metadata.file_modification_time",           # Include file modification timestamp
        current_timestamp().alias("processing_time")  # Add a processing time column
    )
)


# Save the ingested data as a Bronze table in Delta format
(health_raw
 .write
 .format("delta")
 .mode('overwrite')  # Overwrite existing data
 .saveAsTable(f"{my_catalog}.health_data.health_bronze_dev")
)

health_bronze = spark.table(f'{my_catalog}.health_data.health_bronze_dev')


##
## Bronze -> Silver
##

health_silver = (
    health_bronze
    # Create a new column to categorize the HighCholest column
    .withColumn(
        "HighCholest_Group", 
        when(col("HighCholest") == 0, 'Normal')
        .when(col("HighCholest") == 1, 'Above Average')
        .when(col("HighCholest") == 2, 'High')
        .otherwise('Unknown')
    )
    # Create a new column to categorize the Age_Group column
    .withColumn(
        "Age_Group", 
        when(col("Age") <= 9, "0-9")
        .when((col("Age") >= 10) & (col("Age") <= 19), "10-19")
        .when((col("Age") >= 20) & (col("Age") <= 29), "20-29")
        .when((col("Age") >= 30) & (col("Age") <= 39), "30-39")
        .when((col("Age") >= 40) & (col("Age") <= 49), "40-49")
        .when(col("Age") >= 50, "50+")
        .otherwise('Unknown')
    )
    # Drop unnecessary columns (e.g., metadata columns)
    .drop("file_name", "file_modification_time", "processing_time")
)

# Save the transformed data as a Silver table
(health_silver
 .write
 .format("delta")
 .mode("overwrite")  # Overwrite any existing data
 .saveAsTable(f"{my_catalog}.health_data.health_silver_dev")
)



##
## Silver - Gold
##
chol_age_agg = spark.sql(f'''
    CREATE OR REPLACE TABLE {my_catalog}.health_data.chol_age_agg_gold_dev AS
    SELECT 
        HighCholest_Group, 
        Age_Group, 
        count(*) as Total
    FROM {my_catalog}.health_data.health_silver_dev
    GROUP BY HighCholest_Group, Age_Group
''')

## Display the final gold table
spark.table(f'{my_catalog}.health_data.chol_age_agg_gold_dev').display()

In [0]:
def delete_demo03_tables(catalog):
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.health_data.health_bronze_dev")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.health_data.health_silver_dev")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.health_data.chol_age_agg_dev")

delete_demo03_tables(my_catalog)

**C. Modularize the PySpark Code**
Modularizing code improves maintainability, reusability, and collaboration by breaking workflows into smaller, manageable units. This approach reduces redundancy, simplifies debugging, and allows for easier scaling. It also enables better team collaboration, as different team members can work on separate modules without conflicts, making the overall codebase more flexible and easier to manage. 
****
We will modularize the code from above into the following functions:

a. get_health_csv_schema: Defines and returns the schema for the CSV files.

b. read_health_data: Reads the CSV file into a DataFrame with metadata.

c. high_cholest_map and group_ages_map: Converts the numeric values to categorical.

d. save_df_to_delta: Saves the transformed data to a UC table.

e. get_cholest_age_agg: Aggregates the data for the gold table.

NOTE: We will quickly go over the functions below. Focus on the process of modularizing code, not the code itself. PySpark is a prerequisite for the course.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from pyspark.sql.functions import when, col, current_timestamp


##
## Ingest Cloud to Bronze
##

# a. Function to define the schema for the health data CSV
def get_health_csv_schema():
    return StructType([
        StructField("ID", IntegerType(), True),
        StructField("PII", StringType(), True),
        StructField("date", DateType(), True),
        StructField("HighCholest", IntegerType(), True),
        StructField("HighBP", DoubleType(), True),
        StructField("BMI", DoubleType(), True),
        StructField("Age", DoubleType(), True),
        StructField("Education", DoubleType(), True),
        StructField("income", IntegerType(), True)
    ])


# b. Function to read the CSV data into a DataFrame and add metadata columns
def read_health_data(csv_path, schema):
    return (
        spark
        .read
        .format("csv")
        .option("header", "true")  # Use the header row for column names
        .schema(schema)            # Apply the defined schema
        .load(csv_path)            # Load the CSV data
        .select(
            "*",
            "_metadata.file_name",                        # Include file name from metadata
            "_metadata.file_modification_time",           # Include file modification timestamp
            current_timestamp().alias("processing_time")  # Add a processing time column
        )
    )


##
## Mapping Functions for Data Transformation
##

# c. Map the 'HighCholest' column to categories
def high_cholest_map(col_name):
    return (
        when(col(col_name) == 0, 'Normal')
        .when(col(col_name) == 1, 'Above Average')
        .when(col(col_name) == 2, 'High')
        .otherwise('Unknown')
    )


# d. Map the 'Age' column to age groups
def group_ages_map(col_name):
    return (
        when(col(col_name) <= 9, "0-9")
        .when((col(col_name) >= 10) & (col(col_name) <= 19), "10-19")
        .when((col(col_name) >= 20) & (col(col_name) <= 29), "20-29")
        .when((col(col_name) >= 30) & (col(col_name) <= 39), "30-39")
        .when((col(col_name) >= 40) & (col(col_name) <= 49), "40-49")
        .when(col(col_name) >= 50, "50+")
        .otherwise('Unknown')
    )


##
## Save a DataFrame to UC Table
##

# e. Function to save the DataFrame to a UC table
def save_df_to_delta(dataframe, uc_table, mode):
    (dataframe
     .write
     .format("delta")
     .mode(mode)             # Specify the save mode (e.g., 'overwrite', 'append')
     .saveAsTable(uc_table)  # Save the DataFrame as a table
    )


##
## Gold Aggregation
##

# f. Function to create a Gold-level table with aggregated counts
def get_cholest_age_agg(catalog, schema, table_name):
    query = f'''
        CREATE OR REPLACE TABLE {catalog}.{schema}.{table_name} AS
        SELECT 
            HighCholest_Group, 
            Age_Group, 
            count(*) as Total
        FROM {catalog}.{schema}.health_silver_dev
        GROUP BY HighCholest_Group, Age_Group
    '''
    return spark.sql(query)

Use the functions from above to build your ingest, bronze, silver, and gold pipeline.

In [0]:
##
## CSV to Bronze
##

# Read the health CSV data into a DataFrame and save it to the Bronze table
health_csv_df = read_health_data(
    csv_path = f"/Volumes/{my_catalog}/health_data/health", 
    schema = get_health_csv_schema()
)

# Save the raw data as a Bronze table in Delta format
save_df_to_delta(health_csv_df, f"{my_catalog}.health_data.health_bronze_dev", mode="overwrite")


##
## Bronze to Silver
##

# Transform the data by adding new columns and cleaning up metadata
health_bronze = spark.table(f'{my_catalog}.health_data.health_bronze_dev')

silver_df = (
    health_bronze
    .withColumn("HighCholest_Group", high_cholest_map("HighCholest"))    # Categorize HighCholest
    .withColumn("Age_Group", group_ages_map("Age"))                     # Categorize Age
    .drop("file_name", "file_modification_time", "processing_time")     # Drop unnecessary metadata columns
)

# Save the transformed data as a Silver table in Delta format
save_df_to_delta(silver_df, f"{my_catalog}.health_data.health_silver_dev", mode="overwrite")


##
## Gold Table
##

# Aggregate the data at the Gold level by cholesterol and age groups
get_cholest_age_agg(catalog = my_catalog, schema = 'health_data', table_name ='chol_age_agg_gold_dev')

In [0]:
# Display values
spark.table(f'{my_catalog}.health_data.health_bronze_dev').display()
spark.table(f'{my_catalog}.health_data.health_silver_dev').display()
spark.table(f'{my_catalog}.health_data.chol_age_agg_gold_dev').display()